# Pilot source audit

This notebook evaluates feasibility before full collection. It measures evidence coverage, retrieval integrity, decision-parser completeness, the primary coding queue, and source-system gaps. Counts are operational diagnostics—not substantive findings about steward consistency.

In [ ]:
from pathlib import Path

import duckdb
import matplotlib.pyplot as plt
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DB_PATH = ROOT / 'data' / 'processed' / 'f1_stewarding.duckdb'
QUEUE_PATH = ROOT / 'data' / 'manual' / 'pilot_coding_queue.csv'
con = duckdb.connect(str(DB_PATH), read_only=True)

## 1. Archive and retrieval coverage

In [ ]:
inventory = con.sql('''
SELECT e.season, d.event_id, d.document_class,
       count(*) AS advertised, count(d.content_sha256) AS retrieved,
       count(*) FILTER (WHERE d.is_recalled) AS unavailable_recalled
FROM raw.source_documents d
JOIN metadata.events e USING (event_id)
GROUP BY ALL ORDER BY e.season, d.document_class
''').df()
display(inventory)

classes = ['steward_decision', 'summons', 'final_classification']
chart = inventory[inventory.document_class.isin(classes)]
pivot = chart.pivot(index='event_id', columns='document_class', values='advertised').fillna(0)
ax = pivot.plot.bar(figsize=(10, 5), color=['#3949ab', '#00897b', '#ef6c00'])
ax.set(title='Official documents advertised in each pilot archive', xlabel='', ylabel='Documents')
plt.xticks(rotation=0)
plt.tight_layout()

## 2. Parser completeness

A standard-layout document is complete when Fact, Infringement/Offence, Decision, and Reason are extracted. Special administrative decisions may legitimately omit some sections and must be reviewed rather than imputed.

In [ ]:
parser = con.sql('''
SELECT d.event_id, count(*) AS parsed, count(t.fact_text) AS facts,
       count(t.infringement_text) AS infringements,
       count(t.decision_text) AS decisions, count(t.reason_text) AS reasons
FROM raw.document_text t JOIN raw.source_documents d USING (document_id)
GROUP BY d.event_id ORDER BY d.event_id
''').df()
display(parser)
parser.set_index('event_id')[['facts','infringements','decisions','reasons']].plot.bar(
    figsize=(10, 5), color=['#5c6bc0', '#26a69a', '#ffa726', '#8d6e63'])
plt.title('Extracted decision sections by pilot event')
plt.xlabel('')
plt.ylabel('Documents')
plt.xticks(rotation=0)
plt.tight_layout()

## 3. Primary human-review queue

In [ ]:
queue = pd.read_csv(QUEUE_PATH)
display(queue[['season','event_name','driver_name','incident_family_suggestion',
               'outcome_family_suggestion','source_url']])
pd.crosstab(queue.incident_family_suggestion, queue.outcome_family_suggestion, margins=True)

## 4. Pilot interpretation

The pilot is viable for archive discovery and text extraction: all selected linked evidence PDFs were retrieved with checksums, and nearly all decision layouts yielded the core labeled sections. Two recalled 2025 records appear in the official archive without downloadable links; preserving them as unavailable lineage records prevents silent version loss.

The primary queue is intentionally small because it excludes strict-liability and non-driving offences. Before scaling, an analyst must resolve multi-document incidents, recalled/corrected lineage, guideline coding, and the distinction between official fact and manually observed context.

In [ ]:
con.close()